# 07 · Dokumentierte Geräte im GaSi-Inventar anlegen

Mit frischem Kernel im Repository starten. **Run All zeigt eine Vorschau und liest den
Playground-Bestand.** Der Schreibschritt ist standardmäßig deaktiviert.

Ziel ist die eigene Collection `/8.1_EQUIPMENT/GASI/INSTRUMENTS`.
Die zentrale lokale `config/playground_test_target.json` enthält dafür den Abschnitt
`gasi_inventory`. Der tatsächliche Standort benötigt einen gültigen BAM-Code sowie
Bestätigung und Beleg in `location`; der synthetische Standort-Testwert wird nicht übernommen.
Der Arbeitsauftrag steht unter `authorization` und dokumentiert keine formelle BAM-Freigabe.

Es werden ausschließlich belegte Geräteidentitäten als `INSTRUMENT` angelegt oder anhand
Hersteller und Seriennummer wiederverwendet. Keine Messdaten, Belegdateien oder Preise
werden hochgeladen. Aktuelle Positionen und Prüfmittelstatus bleiben separat zu bestätigen.

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display
from gasi_inventory import write_assignment_report, show_assignment_summary
from gasi_inventory_openbis import inspect_inventory

plan, settings, message = inspect_inventory()
print(message)
show_assignment_summary(plan)
print("Inventarziel:", settings["target"])
if plan.get("location_issue"):
    print("Offen:", plan["location_issue"])
if plan["available"]:
    counts = {}
    for row in plan["rows"]:
        status = row.get("registration_status", "noch nicht mit Instanz geprüft")
        counts[status] = counts.get(status, 0) + 1
    print("Geplante Geräteaktionen:", counts)
    html_path = write_assignment_report(plan)
    display(FileLink(str(html_path.relative_to(Path.cwd()))))

## Geräteanlage bewusst starten

Die Vorschau im lokalen HTML-Bericht prüfen. Danach `REGISTER_ENABLED = True`
setzen und die Phrase `REGISTER DOCUMENTED GASI DEVICE IDENTITIES ON PLAYGROUND`
eintragen; nur die nächste Zelle ausführen.

Vor dem Schreiben werden Belege und Dateihashes, Host, Ziel, Pflichtfelder und
Wertelisten erneut geprüft. Bestehende Identitäten werden wiederverwendet;
Konflikte und unvollständige Identitäten bleiben offen. Angelegte Einträge werden
zurückgelesen, ihre openBIS-IDs in die lokale Setup-Datei zurückgeschrieben.
Ein unterbrochener Ablauf kann mit denselben Identitäten fortgesetzt werden.
Immer nur einen schreibenden Kernel verwenden.

In [ ]:
REGISTER_ENABLED = False
REGISTER_CONFIRMATION = ""

if not REGISTER_ENABLED:
    print("Geräteanlage deaktiviert. Nur lokale Vorschau und lesende Instanzprüfung.")
else:
    from gasi_inventory_openbis import register_inventory
    result, message = register_inventory(REGISTER_CONFIRMATION)
    print(message)
    if result is not None:
        html_path = write_assignment_report(result)
        display(FileLink(str(html_path.relative_to(Path.cwd()))))